In [37]:
from openai import OpenAI
import json
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()
from typing import Optional

In [17]:
car_text = """
    The BMW M340i xDrive is a formidable sports sedan that blends everyday luxury with thrilling performance. 
    At its core lies the legendary 3.0-litre B58 TwinPower Turbo inline six-cylinder petrol engine, producing 
    an impressive 374 horsepower and 500 Nm of peak torque. This immense power is seamlessly directed to all 
    four wheels via an ultra-fast 8-speed Steptronic Sport automatic transmission and an intelligent, rear-biased 
    xDrive all-wheel-drive system. Designed to dominate corners, the sedan sits on a 10mm lowered Adaptive M Suspension 
    that works dynamically alongside an M Sport rear differential and variable sport steering to guarantee sharp, 
    razor-focused handling precision. When launching from a standstill, the M340i xDrive rockets from 0 to 100 kmph 
    in a blistering 4.4 seconds before continuing on to an electronically limited top speed of 250 kmph. Rolling out 
    on aggressive 19-inch M light alloy wheels backed by high-performance M Sport brakes, this medium-sized premium 
    saloon registers a length of 4,714 mm, features a cargo capacity of 480 litres, and boasts an advanced interior 
    highlighted by a seamless curved display carrying BMW OS 8.5.
    """

# using pydantic models to enforce structured output from LLM

In [49]:
class CarSpecs(BaseModel):
    # --- REQUIRED IDENTITY & CORE MECHANICALS ---
    model_name: str = Field(description="The exact name of the car model.")
    engine_type: str = Field(description="Engine designation and cylinder layout.")
    horsepower: int = Field(description="Horsepower rating as an integer.")
    torque_nm: int = Field(description="Torque in Newton-meters as an integer.")
    transmission: str = Field(description="The type of transmission used.")
    
    # --- OPTIONAL PARAMETERS (Everything after transmission) ---
    drivetrain: Optional[str] = Field(
        default=None, 
        description="The drivetrain setup (e.g. AWD, RWD, xDrive). Return null if missing."
    )
    suspension: Optional[str] = Field(
        default=None, 
        description="Type of suspension setup. Return null if missing."
    )
    acceleration_0_to_100_kmph: Optional[float] = Field(
        default=None, 
        description="0 to 100 kmph time in seconds. Return null if missing."
    )
    top_speed_kmph: Optional[int] = Field(
        default=None, 
        description="Top speed in kmph as an integer. Return null if missing."
    )
    wheel_size_inches: Optional[int] = Field(
        default=None, 
        description="Wheel size diameter in inches as an integer. Return null if missing."
    )
    length_mm: Optional[int] = Field(
        default=None, 
        description="Total length of the car in mm as an integer. Return null if missing."
    )
    cargo_capacity_liters: Optional[int] = Field(
        default=None, 
        description="Trunk space/cargo capacity in liters as an integer. Return null if missing."
    )
    on_road_price: Optional[int] = Field(
        default=None, 
        description="Extract on road price as an integer if explicitly present, otherwise return null."
    )


In [50]:
openai_client = OpenAI()

In [58]:
# pass response_format argument
response = openai_client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract mechanical and performance parameters from the text into the required schema."},
        {"role": "user", "content": car_text}
    ],
    response_format=CarSpecs,
)

In [52]:
specs = response.choices[0].message.parsed

In [56]:
specs

CarSpecs(model_name='BMW M340i xDrive', engine_type='3.0-litre B58 TwinPower Turbo inline six-cylinder', horsepower=374, torque_nm=500, transmission='8-speed Steptronic Sport automatic', drivetrain='xDrive', suspension='Adaptive M Suspension', acceleration_0_to_100_kmph=4.4, top_speed_kmph=250, wheel_size_inches=19, length_mm=4714, cargo_capacity_liters=480, on_road_price=None)

In [55]:
specs.__dict__

{'model_name': 'BMW M340i xDrive',
 'engine_type': '3.0-litre B58 TwinPower Turbo inline six-cylinder',
 'horsepower': 374,
 'torque_nm': 500,
 'transmission': '8-speed Steptronic Sport automatic',
 'drivetrain': 'xDrive',
 'suspension': 'Adaptive M Suspension',
 'acceleration_0_to_100_kmph': 4.4,
 'top_speed_kmph': 250,
 'wheel_size_inches': 19,
 'length_mm': 4714,
 'cargo_capacity_liters': 480,
 'on_road_price': None}

# using json to enforce LLM ouput as a json

In [61]:
# pass response_format argument
response = openai_client.beta.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract mechanical and performance parameters from the text."
        "Output the results as a raw JSON object."},
        {"role": "user", "content": car_text}
    ],
    response_format={"type": "json_object"},
)

In [64]:
print(response.choices[0].message.content)

{
  "mechanical_parameters": {
    "engine": {
      "type": "3.0-litre B58 TwinPower Turbo inline six-cylinder",
      "power": "374 horsepower",
      "torque": "500 Nm"
    },
    "transmission": {
      "type": "8-speed Steptronic Sport automatic"
    },
    "drivetrain": {
      "type": "xDrive all-wheel-drive",
      "bias": "rear-biased"
    },
    "suspension": {
      "type": "Adaptive M Suspension",
      "lowered": "10mm"
    },
    "differential": {
      "type": "M Sport rear differential"
    },
    "steering": {
      "type": "variable sport steering"
    },
    "brakes": {
      "type": "M Sport brakes"
    },
    "wheels": {
      "size": "19-inch M light alloy"
    }
  },
  "performance_parameters": {
    "acceleration": {
      "0_to_100_kmph": "4.4 seconds"
    },
    "top_speed": {
      "electronically_limited": "250 kmph"
    }
  },
  "dimensions": {
    "length": "4,714 mm",
    "cargo_capacity": "480 litres"
  },
  "interior": {
    "display": {
      "type": "

# use json schema to produce exact same json llm output as schema

In [65]:
car_json_schema = {
    "type": "object",
    "properties": {
        "model_name": {"type": "string"},
        "engine_type": {"type": "string"},
        "horsepower": {"type": "integer"},
        "torque_nm": {"type": "integer"},
        "transmission": {"type": "string"},
        "drivetrain": {
            "anyOf": [{"type": "string"}, {"type": "null"}]
        },
        "top_speed_kmph": {
            "anyOf": [{"type": "integer"}, {"type": "null"}]
        }
    },
    "required": [
        "model_name", "engine_type", "horsepower", "torque_nm", 
        "transmission", "drivetrain", "top_speed_kmph"
    ],
    "additionalProperties": False
}

In [77]:
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract car parameters based on the json schema structure."},
        {"role": "user", "content": car_text}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "CarExtractionSchema",
            "schema": car_json_schema,
            "strict": True
        }
    }
)

In [78]:
json.loads(response.choices[0].message.content)

{'model_name': 'BMW M340i xDrive',
 'engine_type': '3.0-litre B58 TwinPower Turbo inline six-cylinder petrol',
 'horsepower': 374,
 'torque_nm': 500,
 'transmission': '8-speed Steptronic Sport automatic',
 'drivetrain': 'xDrive',
 'top_speed_kmph': 250}

# passing temperature = 0 and fixed seed value

In [79]:
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    seed=42,
    messages=[
        {"role": "system", "content": "Extract car parameters based on the json schema structure."},
        {"role": "user", "content": car_text}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "CarExtractionSchema",
            "schema": car_json_schema,
            "strict": True
        }
    }
)

In [83]:
json.loads(response.choices[0].message.content)

{'model_name': 'BMW M340i xDrive',
 'engine_type': '3.0-litre B58 TwinPower Turbo inline six-cylinder petrol',
 'horsepower': 374,
 'torque_nm': 500,
 'transmission': '8-speed Steptronic Sport automatic',
 'drivetrain': 'xDrive',
 'top_speed_kmph': 250}

### Gold standard is using pydantic models